In [ ]:
!rm -rf /content/drive

# ⚙️ BƯỚC 4: Feature Engineering — Google Colab
**Amazon Clothing Review Analysis & Recommendation**

---
### 🎯 Mục tiêu Bước 4:
Chuẩn bị đặc trưng (features) cho các bước sau — **KHÔNG train model**

```
Phần A: Text Features      → HashingVectorizer trên 62M reviews
Phần B: Item Features      → TF-IDF trên meta (title + description)
Phần C: Interaction Matrix → Sparse matrix user × item
```

### 📁 File đầu ra:
```
outputs/models/
├── review_hashing_vectorizer.pkl  ← dùng lại ở Bước 5
├── review_features_train.npz      ← feature matrix train
├── review_features_test.npz       ← feature matrix test
├── review_labels_train.npy        ← nhãn sentiment train
├── review_labels_test.npy         ← nhãn sentiment test
├── item_vectorizer.pkl
├── item_tfidf_matrix.npz
├── item_features.parquet
├── item_id_to_idx.pkl
├── train_matrix.npz
├── test_matrix.npz
├── rec_train.parquet
├── rec_test.parquet
├── user_encoder.pkl
├── item_encoder.pkl
└── rec_metadata.json
```

## Cell 0 — Cài thư viện & Mount Drive

In [2]:
!pip install pyarrow nltk scikit-learn -q

from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted!')

Mounted at /content/drive
✅ Drive mounted!


## Cell 1 — Import & Setup

In [3]:
import pandas as pd
import numpy as np
import pickle, json, re, gc
from pathlib import Path
from scipy import sparse
import pyarrow.parquet as pq
import pyarrow as pa
import psutil

from sklearn.feature_extraction.text import HashingVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
STOPWORDS     = list(stopwords.words('english'))
STOPWORDS_SET = set(STOPWORDS)

def ram_usage():
    vm = psutil.virtual_memory()
    return f'RAM: {vm.used/(1024**3):.1f}/{vm.total/(1024**3):.1f} GB ({vm.percent:.0f}%)'

# ============================================================
# ĐƯỜNG DẪN
# ============================================================
DRIVE_ROOT    = Path('/content/drive/MyDrive/amazon_clothing')
PROCESSED_DIR = DRIVE_ROOT / 'data' / 'processed'
MODELS_DIR    = DRIVE_ROOT / 'outputs' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Label mapping cho sentiment
LABEL_MAP     = {'negative': 0, 'neutral': 1, 'positive': 2}

# Kiểm tra file
print('🔍 Kiểm tra file:')
for fname in ['review_clean.parquet', 'meta_clean.parquet', 'review_for_rec.parquet']:
    f = PROCESSED_DIR / fname
    sz = f.stat().st_size/(1024**3) if f.exists() else 0
    print(f'   {"✅" if f.exists() else "❌"} {fname:<35} {sz:.2f} GB')

print(f'\n💾 {ram_usage()}')

🔍 Kiểm tra file:
   ✅ review_clean.parquet                8.29 GB
   ✅ meta_clean.parquet                  1.72 GB
   ✅ review_for_rec.parquet              3.48 GB

💾 RAM: 1.2/12.7 GB (12%)


---
# PHẦN A — Text Features cho Sentiment Analysis

**Mục tiêu:** Chuyển text reviews thành vector số học
- Dùng **HashingVectorizer** → xử lý chunking được, không cần vocabulary
- Lưu feature matrix train/test để Bước 5 load và train model

```
review_clean.parquet (62M)
        ↓ đọc từng chunk
Làm sạch text
        ↓
HashingVectorizer.transform()
        ↓
Lưu dần vào train/test .npz
```

## Cell 2 — Hàm làm sạch text

In [ ]:
def clean_review_text(text):
    """Làm sạch text review cho NLP"""
    if pd.isna(text): return ''
    text  = str(text).lower()
    text  = re.sub(r'[^a-z\s]', ' ', text)
    text  = re.sub(r'\s+', ' ', text).strip()
    words = [w for w in text.split()
             if w not in STOPWORDS_SET and len(w) > 2]
    return ' '.join(words)

# Test thử
test = "This shirt fits perfectly! Very comfortable and great quality."
print(f'Input : {test}')
print(f'Output: {clean_review_text(test)}')
print(' Clean function ready!')

Input : This shirt fits perfectly! Very comfortable and great quality.
Output: shirt fits perfectly comfortable great quality
 Clean function ready!


## Cell 3 — Khởi tạo HashingVectorizer

In [ ]:
# ============================================================
# HashingVectorizer:
# n_features=2^18 = 262,144 → đủ lớn, tránh hash collision
# ngram_range=(1,2) → unigram + bigram
# norm='l2' → chuẩn hóa vector
# alternate_sign=False → tất cả giá trị dương
# ============================================================

hashing_vectorizer = HashingVectorizer(
    n_features     = 2**18,
    ngram_range    = (1, 2),
    norm           = 'l2',
    alternate_sign = False,
    strip_accents  = 'unicode',
    stop_words     = STOPWORDS
)

print('✅ HashingVectorizer khởi tạo xong!')
print(f'   n_features : {hashing_vectorizer.n_features:,}')
print(f'   ngram_range: {hashing_vectorizer.ngram_range}')
print(f'   norm       : {hashing_vectorizer.norm}')

✅ HashingVectorizer khởi tạo xong!
   n_features : 262,144
   ngram_range: (1, 2)
   norm       : l2


## Cell 4 — Transform 62M reviews theo chunks
> ⏱️ Ước tính: 45-60 phút

In [ ]:
# ============================================================
# CELL 4: Transform 62M reviews theo chunks
# - CÓ SUBSAMPLING (lấy mẫu 5%) để tránh OOM
# - CÓ TQDM để theo dõi tiến trình trực quan
# ============================================================

from tqdm.auto import tqdm # Import thư viện thanh tiến trình

SPLIT_YEAR    = 2022
SAMPLE_FRAC   = 0.05       # Tỷ lệ lấy mẫu 5%

pf            = pq.ParquetFile(PROCESSED_DIR / 'review_clean.parquet')
n_row_groups  = pf.metadata.num_row_groups

train_chunks_X, train_chunks_y = [], []
test_chunks_X,  test_chunks_y  = [], []

total_train = 0
total_test  = 0

print(f'🚀 Bắt đầu transform {n_row_groups} row groups...')
print(f'   Tỷ lệ lấy mẫu: {SAMPLE_FRAC*100}% (Giảm tải RAM)')
print(f'   Split year: {SPLIT_YEAR} (train < {SPLIT_YEAR}, test >= {SPLIT_YEAR})\n')

# Sử dụng tqdm bao bọc vòng lặp range()
pbar = tqdm(range(n_row_groups), desc="Transforming", unit=" chunk")

for rg_idx in pbar:
    # 1. Đọc chunk
    chunk = pf.read_row_group(
        rg_idx,
        columns=['text', 'sentiment', 'year']
    ).to_pandas()

    chunk = chunk.dropna(subset=['text', 'sentiment', 'year'])

    # 2. Lấy mẫu ngẫu nhiên (Subsampling) để cứu RAM
    if len(chunk) > 0:
        chunk = chunk.sample(frac=SAMPLE_FRAC, random_state=rg_idx)

    if len(chunk) == 0:
        continue

    # 3. Làm sạch text
    X_text = chunk['text'].apply(clean_review_text)
    y      = chunk['sentiment'].map(LABEL_MAP).values
    years  = chunk['year'].values

    # Bỏ dòng label không hợp lệ (NaN sau khi map)
    valid  = ~np.isnan(y.astype(float))
    X_text = X_text[valid]
    y      = y[valid].astype(np.int8)
    years  = years[valid]

    # Bỏ qua nếu sau khi dọn dẹp không còn dòng nào
    if len(X_text) == 0:
        continue

    # 4. Transform text → sparse vector bằng HashingVectorizer
    X = hashing_vectorizer.transform(X_text)

    # ✅ FIX: chuyển sang numpy bool array trước khi index
    train_mask = (years < SPLIT_YEAR).astype(bool)
    test_mask  = (years >= SPLIT_YEAR).astype(bool)

    if train_mask.sum() > 0:
        train_chunks_X.append(X[train_mask])
        train_chunks_y.append(y[train_mask])
        total_train += int(train_mask.sum())

    if test_mask.sum() > 0:
        test_chunks_X.append(X[test_mask])
        test_chunks_y.append(y[test_mask])
        total_test += int(test_mask.sum())

    # 6. Dọn dẹp triệt để RAM
    del chunk, X_text, X, y, years, train_mask, test_mask
    gc.collect()

    # 7. Cập nhật thông số lên thanh tiến trình tqdm
    # Trích xuất số GB RAM hiện tại từ hàm ram_usage() để in ra cho gọn
    current_ram = ram_usage().replace('RAM: ', '').split(' ')[0]

    pbar.set_postfix({
        'Train': f'{total_train:,}',
        'Test': f'{total_test:,}',
        'RAM': current_ram
    })

pbar.close() # Đóng thanh tiến trình khi hoàn tất

print(f'\n✅ Transform hoàn tất!')
print(f'   Tổng Train: {total_train:,} samples')
print(f'   Tổng Test : {total_test:,} samples')
print(f'   Trạng thái: {ram_usage()}')

🚀 Bắt đầu transform 661 row groups...
   Tỷ lệ lấy mẫu: 5.0% (Giảm tải RAM)
   Split year: 2022 (train < 2022, test >= 2022)



Transforming:   0%|          | 0/661 [00:00<?, ? chunk/s]

NameError: name 'clean_review_text' is not defined

## Cell 5 — Ghép chunks & Lưu feature matrix

In [ ]:
# ============================================================
# CELL 5: Ghép tất cả chunks thành 1 matrix và lưu
# ============================================================

print('🔄 Ghép train chunks...')
X_train = sparse.vstack(train_chunks_X, format='csr')
y_train = np.concatenate(train_chunks_y)
del train_chunks_X, train_chunks_y
gc.collect()

print('🔄 Ghép test chunks...')
X_test  = sparse.vstack(test_chunks_X, format='csr')
y_test  = np.concatenate(test_chunks_y)
del test_chunks_X, test_chunks_y
gc.collect()

print(f'\n📊 Feature matrices:')
print(f'   X_train: {X_train.shape} | {X_train.data.nbytes/(1024**3):.2f} GB')
print(f'   X_test : {X_test.shape}  | {X_test.data.nbytes/(1024**3):.2f} GB')
print(f'   {ram_usage()}')

# Lưu
print('\n💾 Đang lưu...')
sparse.save_npz(str(MODELS_DIR / 'review_features_train.npz'), X_train)
print('✅ Saved: review_features_train.npz')

sparse.save_npz(str(MODELS_DIR / 'review_features_test.npz'), X_test)
print('✅ Saved: review_features_test.npz')

np.save(str(MODELS_DIR / 'review_labels_train.npy'), y_train)
print('✅ Saved: review_labels_train.npy')

np.save(str(MODELS_DIR / 'review_labels_test.npy'), y_test)
print('✅ Saved: review_labels_test.npy')

# Lưu vectorizer
with open(MODELS_DIR / 'review_hashing_vectorizer.pkl', 'wb') as f:
    pickle.dump(hashing_vectorizer, f)
print('✅ Saved: review_hashing_vectorizer.pkl')

# Lưu label mapping
with open(MODELS_DIR / 'label_map.json', 'w') as f:
    json.dump(LABEL_MAP, f)
print('✅ Saved: label_map.json')

# Phân bố nhãn
print('\n📊 Phân bố nhãn train:')
for label, name in [(0,'negative'),(1,'neutral'),(2,'positive')]:
    cnt = (y_train == label).sum()
    print(f'   {name:10}: {cnt:>10,} ({cnt/len(y_train)*100:.1f}%)')

del X_train, X_test, y_train, y_test
gc.collect()
print(f'\n✅ Phần A hoàn tất! {ram_usage()}')

🔄 Ghép train chunks...
🔄 Ghép test chunks...

📊 Feature matrices:
   X_train: (2488797, 262144) | 0.48 GB
   X_test : (630290, 262144)  | 0.13 GB
   RAM: 3.7/12.7 GB (32%)

💾 Đang lưu...
✅ Saved: review_features_train.npz
✅ Saved: review_features_test.npz
✅ Saved: review_labels_train.npy
✅ Saved: review_labels_test.npy
✅ Saved: review_hashing_vectorizer.pkl
✅ Saved: label_map.json

📊 Phân bố nhãn train:
   negative  :    345,882 (13.9%)
   neutral   :    210,728 (8.5%)
   positive  :  1,932,187 (77.6%)

✅ Phần A hoàn tất! RAM: 2.9/12.7 GB (25%)


---
# PHẦN B — Item Content Features
**Mục tiêu:** Tạo vector đại diện cho từng sản phẩm → dùng cho Content-Based Filtering

## Cell 7 — Load Meta + TF-IDF + Numerical (gộp để tránh mất biến)
> ⚠️ Nếu kernel restart, chạy lại từ Cell 1 → Cell 7 theo thứ tự

In [4]:
# ============================================================
# CELL 7: Load Meta + TF-IDF + Numerical — GỘP LẠI
# Gộp Cell 6 và 7 cũ thành 1 cell để tránh mất biến df_meta
# khi kernel restart giữa chừng
# ============================================================

from tqdm.auto import tqdm
gc.collect()
print(f'💾 Bắt đầu: {ram_usage()}')

# ── BƯỚC 1: Load Meta theo chunks ──────────────────────────
def clean_text(text):
    if pd.isna(text): return ''
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def make_item_content(title, description):
    t = clean_text(title)
    d = clean_text(description)
    return f'{t} {t} {t} {d}'.strip()

meta_pf = pq.ParquetFile(PROCESSED_DIR / 'meta_clean.parquet')
n_rg    = meta_pf.metadata.num_row_groups

all_asins, all_contents = [], []
all_prices, all_avg_rat, all_rat_num = [], [], []

print(f'📦 Loading meta ({n_rg} row groups)...')
for rg_idx in tqdm(range(n_rg), desc='Meta chunks'):
    chunk = meta_pf.read_row_group(
        rg_idx,
        columns=['parent_asin','title','description',
                 'price','average_rating','rating_number']
    ).to_pandas()
    chunk['item_content'] = [
        make_item_content(t, d)
        for t, d in zip(chunk['title'], chunk['description'])
    ]
    chunk = chunk[chunk['item_content'].str.split().str.len() >= 3]
    all_asins.extend(chunk['parent_asin'].tolist())
    all_contents.extend(chunk['item_content'].tolist())
    all_prices.extend(chunk['price'].tolist())
    all_avg_rat.extend(chunk['average_rating'].tolist())
    all_rat_num.extend(chunk['rating_number'].tolist())
    del chunk
    gc.collect()

df_meta = pd.DataFrame({
    'parent_asin'   : all_asins,
    'item_content'  : all_contents,
    'price'         : all_prices,
    'average_rating': all_avg_rat,
    'rating_number' : all_rat_num,
})
del all_asins, all_contents, all_prices, all_avg_rat, all_rat_num
gc.collect()
print(f'✅ Meta loaded: {df_meta.shape} | {ram_usage()}')

# ── BƯỚC 2: Fit TF-IDF ─────────────────────────────────────
# Tham số tối ưu RAM:
#   max_features=5000  → giảm 50% RAM
#   ngram_range=(1,1)  → chỉ unigram → giảm 60% RAM
#   dtype=float32      → giảm 50% RAM matrix
item_vectorizer = TfidfVectorizer(
    max_features  = 5_000,
    ngram_range   = (1, 1),
    min_df        = 10,
    max_df        = 0.95,
    sublinear_tf  = True,
    stop_words    = STOPWORDS,
    strip_accents = 'unicode',
    dtype         = np.float32
)
print('🔄 Fitting TF-IDF...')
item_tfidf = item_vectorizer.fit_transform(df_meta['item_content'])
print(f'✅ TF-IDF: {item_tfidf.shape} | {item_tfidf.data.nbytes/(1024**2):.0f} MB | {ram_usage()}')

# ── BƯỚC 3: Numerical features ─────────────────────────────
df_item_num               = df_meta[['parent_asin']].copy()
df_item_num['price']      = pd.to_numeric(
    df_meta['price'], errors='coerce'
).clip(0, 500).fillna(25.0)
df_item_num['price_tier'] = pd.cut(
    df_item_num['price'],
    bins=[0,25,75,150,500], labels=[0,1,2,3]
).astype(float).fillna(0)
df_item_num['avg_rating'] = pd.to_numeric(
    df_meta['average_rating'], errors='coerce'
).fillna(4.0)
df_item_num['log_rating_count'] = np.log1p(
    pd.to_numeric(df_meta['rating_number'], errors='coerce').fillna(0)
)
scaler   = MinMaxScaler()
num_cols = ['price','price_tier','avg_rating','log_rating_count']
df_item_num[num_cols] = scaler.fit_transform(df_item_num[num_cols])
print(f'✅ Numerical: {df_item_num.shape}')

# ── BƯỚC 4: Lưu item_id_to_idx TRƯỚC khi del df_meta ───────
item_id_to_idx = {asin: idx for idx, asin in enumerate(df_meta['parent_asin'])}

del df_meta
gc.collect()
print(f'✅ Xong! {ram_usage()}')


💾 Bắt đầu: RAM: 1.6/12.7 GB (15%)
📦 Loading meta (145 row groups)...


Meta chunks:   0%|          | 0/145 [00:00<?, ?it/s]

✅ Meta loaded: (7218109, 5) | RAM: 6.3/12.7 GB (52%)
🔄 Fitting TF-IDF...
✅ TF-IDF: (7218109, 5000) | 743 MB | RAM: 7.8/12.7 GB (64%)
✅ Numerical: (7218109, 5)
✅ Xong! RAM: 6.8/12.7 GB (56%)


## Cell 8 — Lưu Item Features

In [5]:
# ============================================================
# CELL 8: Lưu Item Features
# item_id_to_idx đã được tạo ở Cell 7 trước khi del df_meta
# ============================================================

with open(MODELS_DIR / 'item_vectorizer.pkl', 'wb') as f:
    pickle.dump(item_vectorizer, f)
print('✅ Saved: item_vectorizer.pkl')

sparse.save_npz(str(MODELS_DIR / 'item_tfidf_matrix.npz'), item_tfidf)
print('✅ Saved: item_tfidf_matrix.npz')

df_item_num.to_parquet(MODELS_DIR / 'item_features.parquet', index=False)
print('✅ Saved: item_features.parquet')

# item_id_to_idx đã được tạo ở Cell 7
with open(MODELS_DIR / 'item_id_to_idx.pkl', 'wb') as f:
    pickle.dump(item_id_to_idx, f)
print('✅ Saved: item_id_to_idx.pkl')

del item_tfidf, df_item_num, item_id_to_idx
gc.collect()
print(f'\n✅ Phần B hoàn tất! {ram_usage()}')


✅ Saved: item_vectorizer.pkl
✅ Saved: item_tfidf_matrix.npz
✅ Saved: item_features.parquet
✅ Saved: item_id_to_idx.pkl

✅ Phần B hoàn tất! RAM: 4.3/12.7 GB (36%)


---
# PHẦN C — Interaction Matrix cho Collaborative Filtering
**Mục tiêu:** Tạo sparse matrix user × item để train SVD/ALS/NCF

## Cell 9 — Load & Encode

In [6]:
print('📦 Loading review_for_rec.parquet...')
df_rec = pq.read_table(
    PROCESSED_DIR / 'review_for_rec.parquet',
    columns=['user_id','parent_asin','rating','timestamp']
).to_pandas()

print(f'✅ Loaded: {df_rec.shape}')
print(f'   Users  : {df_rec["user_id"].nunique():,}')
print(f'   Items  : {df_rec["parent_asin"].nunique():,}')

# Encode user_id và item_id → integer index
user_encoder          = LabelEncoder()
item_encoder          = LabelEncoder()
df_rec['user_idx']    = user_encoder.fit_transform(df_rec['user_id'])
df_rec['item_idx']    = item_encoder.fit_transform(df_rec['parent_asin'])

n_users = df_rec['user_idx'].nunique()
n_items = df_rec['item_idx'].nunique()
print(f'\n✅ Encoded: {n_users:,} users | {n_items:,} items')
print(f'   {ram_usage()}')

📦 Loading review_for_rec.parquet...
✅ Loaded: (26150403, 4)
   Users  : 3,195,711
   Items  : 1,260,717

✅ Encoded: 3,195,711 users | 1,260,717 items
   RAM: 7.5/12.7 GB (62%)


## Cell 10 — Train/Test Split & Sparse Matrix

In [7]:
# Train/Test split theo thời gian
df_rec['date'] = pd.to_datetime(df_rec['timestamp'], unit='ms', errors='coerce')
df_rec['year'] = df_rec['date'].dt.year

SPLIT_YEAR = 2022
df_train   = df_rec[df_rec['year'] <  SPLIT_YEAR].copy()
df_test    = df_rec[df_rec['year'] >= SPLIT_YEAR].copy()

print(f'✅ Train/Test split:')
print(f'   Train: {len(df_train):,} ({len(df_train)/len(df_rec)*100:.1f}%)')
print(f'   Test : {len(df_test):,} ({len(df_test)/len(df_rec)*100:.1f}%)')

# Tạo sparse matrix
train_matrix = sparse.csr_matrix(
    (df_train['rating'].values,
     (df_train['user_idx'].values, df_train['item_idx'].values)),
    shape=(n_users, n_items)
)
test_matrix = sparse.csr_matrix(
    (df_test['rating'].values,
     (df_test['user_idx'].values, df_test['item_idx'].values)),
    shape=(n_users, n_items)
)

print(f'\n✅ Sparse matrices:')
print(f'   Train: {train_matrix.shape} | {train_matrix.nnz:,} non-zeros')
print(f'   Test : {test_matrix.shape} | {test_matrix.nnz:,} non-zeros')
print(f'   Density: {train_matrix.nnz/(n_users*n_items)*100:.6f}%')
print(f'   {ram_usage()}')

✅ Train/Test split:
   Train: 20,551,232 (78.6%)
   Test : 5,599,171 (21.4%)

✅ Sparse matrices:
   Train: (3195711, 1260717) | 20,545,981 non-zeros
   Test : (3195711, 1260717) | 5,598,114 non-zeros
   Density: 0.000510%
   RAM: 8.4/12.7 GB (69%)


## Cell 11 — Lưu Interaction Matrix & Tổng kết

In [8]:
# Lưu encoders
with open(MODELS_DIR / 'user_encoder.pkl', 'wb') as f:
    pickle.dump(user_encoder, f)
with open(MODELS_DIR / 'item_encoder.pkl', 'wb') as f:
    pickle.dump(item_encoder, f)
print('✅ Saved: user_encoder.pkl & item_encoder.pkl')

# Lưu sparse matrices
sparse.save_npz(str(MODELS_DIR / 'train_matrix.npz'), train_matrix)
sparse.save_npz(str(MODELS_DIR / 'test_matrix.npz'),  test_matrix)
print('✅ Saved: train_matrix.npz & test_matrix.npz')

# Lưu train/test data
df_train[['user_idx','item_idx','rating']].to_parquet(
    MODELS_DIR / 'rec_train.parquet', index=False
)
df_test[['user_idx','item_idx','rating']].to_parquet(
    MODELS_DIR / 'rec_test.parquet', index=False
)
print('✅ Saved: rec_train.parquet & rec_test.parquet')

# Lưu metadata
metadata = {
    'n_users'    : int(n_users),
    'n_items'    : int(n_items),
    'n_train'    : int(len(df_train)),
    'n_test'     : int(len(df_test)),
    'split_year' : SPLIT_YEAR,
    'sparsity'   : float(1 - train_matrix.nnz / (n_users * n_items))
}
with open(MODELS_DIR / 'rec_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('✅ Saved: rec_metadata.json')

# Tổng kết
print('\n' + '='*60)
print('🎉 BƯỚC 4 HOÀN TẤT!')
print('='*60)
print('\n📁 Files đã lưu trên Drive:')
for f in sorted(MODELS_DIR.glob('*')):
    sz   = f.stat().st_size
    unit = 'GB' if sz > 1024**3 else 'MB' if sz > 1024**2 else 'KB'
    val  = sz/(1024**3) if sz > 1024**3 else sz/(1024**2) if sz > 1024**2 else sz/1024
    print(f'   {f.name:<45} {val:.1f} {unit}')

print(f'\n📌 Bước tiếp theo: Bước 5 — Sentiment Analysis')
print(f'   Load: review_features_train.npz + review_labels_train.npy')
print(f'   Train: SGD, Logistic Regression, Random Forest, BERT...')

✅ Saved: user_encoder.pkl & item_encoder.pkl
✅ Saved: train_matrix.npz & test_matrix.npz
✅ Saved: rec_train.parquet & rec_test.parquet
✅ Saved: rec_metadata.json

🎉 BƯỚC 4 HOÀN TẤT!

📁 Files đã lưu trên Drive:
   item_encoder.pkl                              15.6 MB
   item_features.parquet                         90.6 MB
   item_id_to_idx.pkl                            123.8 MB
   item_tfidf_matrix.npz                         994.9 MB
   item_vectorizer.pkl                           189.6 KB
   label_map.json                                0.0 KB
   rec_metadata.json                             0.1 KB
   rec_test.parquet                              40.6 MB
   rec_train.parquet                             140.2 MB
   review_features_test.npz                      44.8 MB
   review_features_train.npz                     168.1 MB
   review_hashing_vectorizer.pkl                 1.8 KB
   review_labels_test.npy                        615.6 KB
   review_labels_train.npy                    